**Laboratorio 10**

María Paula Angulo Collantes

Danna Gabriela Castro

**Punto 1**

In [3]:
import pandas as pd
import numpy as np

df_potencial = pd.read_csv('FinAcceso_mercado_potencial.csv')
df_solicitudes = pd.read_csv('FinAcceso_solicitudes_credito.csv')

print("Potencial")
display(df_potencial)

print("Solicitudes")
display(df_solicitudes.head())

Potencial


,zona,hogares_potenciales,descripcion_fuente
0,Norte,1400,Estimación administrativa simulada
1,Centro,2200,Estimación administrativa simulada
2,Sur,3700,Estimación administrativa simulada
3,Periferia,2700,Estimación administrativa simulada


Solicitudes


,id_solicitud,fecha_solicitud,zona,canal,ingreso_mensual_cop,historial_crediticio,puntaje_riesgo,carga_financiera,monto_solicitado_cop,decision
0,FA-00003,2026-07-01,Sur,Digital,2684000,Limitado,598,0.176,5800000,Rechazada
1,FA-00024,2026-07-01,Sur,Digital,1671000,Amplio,546,0.622,980000,Rechazada
2,FA-00035,2026-07-01,Centro,Digital,3984000,Amplio,753,0.590,2360000,Aprobada
3,FA-00460,2026-07-01,Norte,Digital,3362000,Amplio,605,0.620,4990000,Aprobada
4,FA-00661,2026-07-01,Periferia,Digital,1715000,Limitado,566,0.437,1680000,Rechazada


**Supuestos Implícitos en la Recomendación:**

**1.** Supuesto de Demanda Relevada: Asume que el número de solicitudes y aprobaciones observadas refleja fielmente la demanda real y la necesidad de microcréditos en la población.

**2.** Supuesto de Eficiencia Operativa: Asume que concentrar la asesoría donde hay más aprobaciones maximiza el aprovechamiento del presupuesto, ignorando que el bajo desempeño en zonas rezagadas puede ser consecuencia directa de la falta de acompañamiento inicial.

**Necesidad de Verificación:**

Si los datos observados sufren de sesgo de selección o mecanismo de captura desigual, optimizar sobre estas métricas perpetuará la exclusión financiera y dejará invisibilizados a grupos con alta necesidad real.

**Variables:**

FinAcceso_mercado_potencial.csv: Describe la población real de interés (estimación de hogares potencialmente interesados por zona).

FinAcceso_solicitudes_credito.csv: Describe únicamente a la muestra observada(personas que lograron o decidieron radicar una solicitud).

**Grupo Invisible:** Aquellos hogares del mercado potencial que nunca presentaron una solicitud debido a barreras de conectividad, falta de canal presencial o ausencia de asesoría previa.

**Pregunta Concreta de Auditoría:**

¿Existen diferencias significativas en la tasa de captura de solicitudes y en las tasas de aprobación por zona al controlar por el canal de entrada y el potencial real de hogares?

**Punto 2**

Unimos ambas fuentes por la variable zona para comparar la distribución del mercado potencial frente a la captura real de solicitudes.

In [4]:
solicitudes_por_zona = df_solicitudes.groupby('zona').size().reset_index(name='solicitudes_recibidas')
df_zona_merged = pd.merge(df_potencial, solicitudes_por_zona, on='zona', how='outer')

total_solicitudes = df_zona_merged['solicitudes_recibidas'].sum()
df_zona_merged['pct_total_solicitudes'] = (df_zona_merged['solicitudes_recibidas'] / total_solicitudes) * 100
df_zona_merged['solicitudes_por_1000_hogares'] = (df_zona_merged['solicitudes_recibidas'] / df_zona_merged['hogares_potenciales']) * 1000

tabla_punto2 = df_zona_merged[['zona', 'hogares_potenciales', 'solicitudes_recibidas', 'pct_total_solicitudes', 'solicitudes_por_1000_hogares']].copy()
tabla_punto2.columns = ['Zona', 'Hogares Potenciales', 'Solicitudes Recibidas', '% Total Solicitudes', 'Solicitudes por 1.000 Hogares']

display(tabla_punto2.style.format({
    'Hogares Potenciales': '{:,.0f}',
    'Solicitudes Recibidas': '{:,.0f}',
    '% Total Solicitudes': '{:.2f}%',
    'Solicitudes por 1.000 Hogares': '{:.2f}'
}))

,Zona,Hogares Potenciales,Solicitudes Recibidas,% Total Solicitudes,Solicitudes por 1.000 Hogares
0,Centro,"2,200",395,29.92%,179.55
1,Norte,"1,400",475,35.98%,339.29
2,Periferia,"2,700",132,10.00%,48.89
3,Sur,"3,700",318,24.09%,85.95


Mercado Potencial vs. Necesidad

Tener menos solicitudes no implica menor necesidad del servicio:

La zona sur posee el mercado potencial más amplio (3.700 hogares, 37\% del total de la ciudad), pero únicamente genera 85,95 solicitudes por cada 1.000 hogares.

En contraste, el norte (con solo $1.400$ hogares potenciales) capta 339,29 solicitudes por cada 1.000 hogares.

Esto confirma la existencia de un sesgo de selección/captura. La baja cantidad de solicitudes en Sur y Periferia refleja barreras de acceso, no menor demanda real.

**Punto 3**

In [5]:
tasa_global = (df_solicitudes['decision'] == 'Aprobada').mean() * 100
total_aprobadas = (df_solicitudes['decision'] == 'Aprobada').sum()

print(f"Tasa Global de Aprobación: {tasa_global:.2f}% ({total_aprobadas} aprobaciones de {len(df_solicitudes)} solicitudes)\n")

tabla_punto3 = df_solicitudes.groupby('zona').agg(
    solicitudes_recibidas=('id_solicitud', 'count'),
    aprobaciones=('decision', lambda x: (x == 'Aprobada').sum()),
    tasa_aprobacion=('decision', lambda x: (x == 'Aprobada').mean() * 100)
).reset_index()

tabla_punto3.columns = ['Zona', 'Solicitudes Recibidas', 'Aprobaciones', 'Tasa de Aprobación (%)']

display(tabla_punto3.style.format({
    'Solicitudes Recibidas': '{:,.0f}',
    'Aprobaciones': '{:,.0f}',
    'Tasa de Aprobación (%)': '{:.2f}%'
}))

# Exploración de variable adicional: Canal por zona
tabla_canal = pd.crosstab(df_solicitudes['zona'], df_solicitudes['canal'], normalize='index') * 100
print("Porcentaje de Solicitudes por Canal dentro de cada Zona")
display(tabla_canal.style.format('{:.2f}%'))

Tasa Global de Aprobación: 49.85% (658 aprobaciones de 1320 solicitudes)



,Zona,Solicitudes Recibidas,Aprobaciones,Tasa de Aprobación (%)
0,Centro,395,193,48.86%
1,Norte,475,295,62.11%
2,Periferia,132,41,31.06%
3,Sur,318,129,40.57%


Porcentaje de Solicitudes por Canal dentro de cada Zona


canal,Digital,Presencial
zona,,
Centro,81.01%,18.99%
Norte,91.58%,8.42%
Periferia,40.91%,59.09%
Sur,64.15%,35.85%


Análisis del Desempeño y Comparación

En absolutos, el norte lidera con 295 aprobaciones y periferia queda en último lugar con 41.

En tasas de aprobación, la estructura se mantiene (Norte 62,11\% vs. Periferia 31,06\%).

Interpretación Engañosa: Concluir que el norte "atiende mejor" es engañoso. El norte acumula más aprobaciones porque logra capturar más solicitudes iniciales. Si ajustamos la tasa reponderando por el verdadero mercado potencial, la efectividad real sobre la población decrece.

**Variable Adicional Crucial — canal:**

En la periferia, el 59,09\% de las solicitudes provienen del canal Presencial, mientras que en el Norte el 91,58\% ingresa por la vía Digital. Evaluar las tasas de aprobación sin considerar el canal oculta que las barreras operativas de la plataforma digital afectan diferencialmente a las zonas.

**Punto 4**

In [6]:
def categorizar_puntaje(score):
    if score < 610:
        return '1. Menor de 610'
    elif 610 <= score <= 689:
        return '2. 610 a 689'
    else:
        return '3. 690 o más'

df_solicitudes['intervalo_puntaje'] = df_solicitudes['puntaje_riesgo'].apply(categorizar_puntaje)

tabla_punto4 = df_solicitudes.groupby(['intervalo_puntaje', 'historial_crediticio']).agg(
    solicitudes=('id_solicitud', 'count'),
    aprobaciones=('decision', lambda x: (x == 'Aprobada').sum()),
    tasa_aprobacion=('decision', lambda x: (x == 'Aprobada').mean() * 100)
).reset_index()

tabla_punto4.columns = ['Intervalo Puntaje', 'Historial Crediticio', 'Solicitudes', 'Aprobaciones', 'Tasa Aprobación (%)']

display(tabla_punto4.style.format({
    'Solicitudes': '{:,.0f}',
    'Aprobaciones': '{:,.0f}',
    'Tasa Aprobación (%)': '{:.2f}%'
}))

,Intervalo Puntaje,Historial Crediticio,Solicitudes,Aprobaciones,Tasa Aprobación (%)
0,1. Menor de 610,Amplio,187,46,24.60%
1,1. Menor de 610,Limitado,261,40,15.33%
2,2. 610 a 689,Amplio,319,186,58.31%
3,2. 610 a 689,Limitado,171,76,44.44%
4,3. 690 o más,Amplio,318,263,82.70%
5,3. 690 o más,Limitado,64,47,73.44%


**¿Persisten las diferencias al controlar por puntaje?** Sí. En todos los tramos de puntaje, las personas con historial Amplio presentan tasas de aprobación sustancialmente superiores a las de historial Limitado. Por ejemplo, en el tramo medio ($610\text{–}689$), la tasa es de $58,31\%$ (Amplio) vs. $44,44\%$ (Limitado).

**¿Implica esto causalidad o discriminación intencional?** No necesariamente.

**Sesgo de Historia / Proxy:** El historial_crediticio actúa como una variable proxy de la exclusión financiera histórica. En sectores populares e informales, contar con historial Limitado es consecuencia de no haber tenido acceso previo al sistema bancario tradicional, no de una incapacidad real de pago.

**Datos Faltantes a Auditar:** Falta evaluar ingresos informales no declarados, la carga financiera estructurada y la tasa de abandono durante la radicación del crédito.

**Punto 5**

Consecuencias de Recortar la Asesoría Presencial:

**1.** Inmediata: Desplome en la recepción de solicitudes en Periferia y Sur, donde la población depende en un 59,1\% y 35,8\% respectivamente de la atención presencial.
**2.**  a Largo Plazo (Círculo Vicioso):Aumento del sesgo en la base de datos. Al no capturar datos de estas zonas, los futuros modelos concluirán que "no hay mercado" en Periferia/Sur, consolidando la exclusión.

Grupos más Afectados

Hogares de bajos ingresos (Periferia registra un ingreso medio de 1,77M COP vs. Norte con 4,63M COP) y trabajadores informales dependientes del canal presencial.

Estrategias de mitigación de daños

-Calidad y cobertura de Datos: Capturar variables alternativas de capacidad de pago (pago puntual de servicios públicos o arriendos) e integrar encuestas de demanda no atendida en campo.

-Implementación y monitoreo: Implementar un piloto de asesoría móvil/itinerante en Sur y Periferia monitoreando trimestralmente las tasas de conversión por canal, en lugar de un recorte definitivo.

**Punto 6**

Recomendamos aplazae la propuesta de la gerencia. Concentrar la asesoría en el Norte y Centro sesgará la operación hacia zonas que hoy capturan el $65,9\%$ de las solicitudes pero representan solo el $36,0\%$ del mercado real ($3.600$ de $10.000$ hogares). En Periferia, la captura es de apenas $48,89$ solicitudes por cada $1.000$ hogares, debido a barreras en el canal de entrada y no a falta de demanda. Mantenemos como limitación abierta la falta de información sobre el comportamiento de mora e ingresos informales. El paso siguiente es ejecutar un piloto de asesoría presencial móvil en el Sur durante seis meses para medir la demanda latente.

**Reflexión Ética:** Sería irresponsable afirmar públicamente que las zonas periféricas no necesitan crédito o tienen menor demanda, cuando los datos reflejan únicamente una barrera de acceso en nuestro canal de captura.
